# Support Vector Machine (SVM) Optimization in High-Dimensional Data
In this notebook, we analyze the performance of a Support Vector Machine (SVM) on a high-dimensional synthetic dataset. We compare three key strategies to improve model performance:

Regularization (C parameter tuning)
Dimensionality reduction using PCA
Feature selection using statistical tests

We evaluate how each method affects classification accuracy and generalization.

Minimize:

(1/2)||w||² + C Σξᵢ

where:

- ||w||² controls the margin size
- ξᵢ represents classification errors
- C controls the penalty for misclassification

## Import Libraries

In [1]:
from sklearn.datasets import make_classification
from sklearn.decomposition import PCA
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.model_selection import train_test_split
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score
import numpy as np

## Generate High-Dimensional Dataset
Many real-world datasets contain a large number of features.

Challenges associated with high-dimensional data include:

- Increased computational cost
- Risk of overfitting
- Redundant features
- Irrelevant features

In this section, we generate a synthetic dataset with 100 features and evaluate different strategies for improving SVM performance.

This setup simulates a realistic machine learning problem where many features may not provide useful information.

In [2]:
X,y=make_classification(
    n_samples=1000,
    n_features=100,
    n_informative=10,
    n_redundant=40,
    n_repeated=10,
    random_state=42
)

### Train Test Split Dataset

The dataset is divided into training and testing subsets.

- Training data is used to learn the model.
- Testing data is used to evaluate generalization performance.

In [3]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

print("Original feature dimension:", X.shape[1])

Original feature dimension: 100


### Baseline (Higher C Value)

- We first train an RBF SVM using all available features with: **C = 10**
- This model serves as a baseline for comparison with other techniques such as dimensionality reduction and feature selection.
- In SVM, the parameter C controls the penalty applied to classification errors.
- A larger value of C places a higher penalty on misclassified samples, causing the model to focus more on correctly classifying the training data.
- The baseline model uses a relatively large value of C to prioritize classification accuracy on the training set.

**When C increases:**

- Misclassification penalty increases
- Margin becomes smaller
- Model fits training data more closely
- Risk of overfitting increases

In [4]:
svm_baseline=SVC(kernel="rbf",C=10)
svm_baseline.fit(X_train,y_train)

pred_base=svm_baseline.predict(X_test)
acc_base=accuracy_score(y_test,pred_base)

print(f"Baseline Accuracy: {acc_base:.4f}")

Baseline Accuracy: 0.9700


### Regularization (Lower C Value)

- We first train an RBF SVM using all available features with: **C = 10**
- The parameter C controls the trade-off between:
  - Maximizing the margin
  - Minimizing classification errors
- Lower values of C apply stronger regularization.
- This allows the SVM to tolerate some training errors in exchange for a wider margin and a potentially simpler decision boundary.

**When C decreases:**

- Misclassification penalty decreases
- Margin becomes wider
- Model becomes simpler
- Generalization may improve
- Risk of underfitting increases if C is too small

In [5]:
svm_reg=SVC(kernel="rbf",C=1)
svm_reg.fit(X_train,y_train)

pred_reg=svm_reg.predict(X_test)
acc_reg=accuracy_score(y_test,pred_reg)

print(f"Regularized Accuracy: {acc_reg:.4f}")

Regularized Accuracy: 0.9550


### Dimensionality Reduction using PCA

- Principal Component Analysis reduces dimensionality by projecting data onto directions of maximum variance.
- PCA identifies orthogonal directions that capture the greatest variance in the data.
- These directions are called principal components.
- The first component captures the largest variance, the second captures the next largest variance, and so on.

Benefits include:

- Faster training
- Reduced storage requirements
- Noise reduction
- Better generalization in some cases

The original 100-dimensional dataset is reduced to 20 principal components.

In [6]:
pca=PCA(n_components=20)
X_train_pca=pca.fit_transform(X_train)
X_test_pca=pca.transform(X_test)

#### Train SVM on PCA Features

In [7]:
svm_pca=SVC(kernel="rbf",C=1)
svm_pca.fit(X_train_pca,y_train)

pred_pca=svm_pca.predict(X_test_pca)
acc_pca=accuracy_score(y_test,pred_pca)

print(f"PCA Accuracy: {acc_pca:.4f}")

PCA Accuracy: 0.9550


#### Variance Retained by PCA

The explained variance ratio indicates how much information is preserved after dimensionality reduction.

A high value means that most of the original dataset information has been retained.

In [8]:
variance_retained=np.sum(pca.explained_variance_ratio_)
print(f"Variance Retained: {variance_retained:.2%}")

Variance Retained: 96.64%


### Feature Selection (SelectKBest - ANOVA F-test)

Instead of transforming features, feature selection keeps only the most informative features. In this experiment, SelectKBest uses the ANOVA F-test to select the top 20 features.

The F-statistic measures how strongly a feature is related to the target variable.

Higher scores indicate more informative features.

F = Variance Between Classes / Variance Within Classes

In [9]:
selector=SelectKBest(
    score_func=f_classif,
    k=20
)

X_train_sel=selector.fit_transform(X_train, y_train)
X_test_sel=selector.transform(X_test)

#### Train SVM on Selected Features

In [10]:
svm_sel=SVC(kernel="rbf",C=1)
svm_sel.fit(X_train_sel,y_train)

pred_sel=svm_sel.predict(X_test_sel)
acc_sel=accuracy_score(y_test,pred_sel)

print(f"Feature Selection Accuracy: {acc_sel:.4f}")

Feature Selection Accuracy: 0.9350


# Conclusion 
`Baseline SVM may overfit due to high dimensionality` 

`Regularization (lower C) improves generalization`

`PCA reduces noise and improves efficiency`

`Feature Selection keeps only informative feature`

In [11]:
print(f"Baseline SVM Accuracy      : {acc_base:.4f}")
print(f"Regularized SVM Accuracy   : {acc_reg:.4f}")
print(f"PCA + SVM Accuracy         : {acc_pca:.4f}")
print(f"Feature Selection Accuracy : {acc_sel:.4f}")

Baseline SVM Accuracy      : 0.9700
Regularized SVM Accuracy   : 0.9550
PCA + SVM Accuracy         : 0.9550
Feature Selection Accuracy : 0.9350
